In [3]:

from __future__ import annotations
from dataclasses import dataclass
import math
from typing import Literal, Optional
import pynucastro as pyna
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple


In [ ]:
"""
Alpha-decay rates from Song et al. (2022, https://doi.org/10.1140/epja/s10050-022-00898-1)

We convert half-life -> decay constant: lambda = ln(2) / T_1/2

Source: Song et al., Eur. Phys. J. A (2022) 58: 244, https://doi.org/10.1140/epja/s10050-022-00898-1
"""


LN2 = math.log(2.0)


# -----------------------------
# Coefficients from Table 1
# -----------------------------


SONG_COEF = dict(
    a=50.693,
    b=-51.778,
    heo=0.284,
    hoo=0.568
)




# -----------------------------
# Helpers
# -----------------------------
def even_odd_class(Z: int, N: int) -> Literal["ee", "eo", "oe", "oo"]:
    """
    ee: even Z, even N
    eo: even Z, odd  N
    oe: odd  Z, even N
    oo: odd  Z, odd  N
    """
    z_even = (Z % 2 == 0)
    n_even = (N % 2 == 0)
    if z_even and n_even:
        return "ee"
    if z_even and (not n_even):
        return "eo"
    if (not z_even) and n_even:
        return "oe"
    return "oo"


def reduced_mass_amu(A_d: int, A_alpha: int = 4) -> float:
    """
    Paper uses reduced mass mu = A_d*A_alpha/(A_d + A_alpha) (dimensionless in amu units).
    """
    return (A_d * A_alpha) / (A_d + A_alpha)


def lmin_without_parity(j_parent: float, j_daughter: float) -> int:
    """
    REMOVE parity dependence: set lmin = |jp - jd| only.

    Note: jp and jd can be half-integers. |jp-jd| should be an integer for physical decays,
    but to be robust we round to nearest integer and verify it's close.
    """
    dj = abs(j_parent - j_daughter)
    l = int(round(dj))
   
    if l < 0:
        raise ValueError("Computed l < 0 (should never happen).")
    return l


@dataclass(frozen=True)
class AlphaDecayInput:
    """
    Parent -> Daughter + alpha.
    Provide Q_alpha in MeV, and parent/daughter spins (jp, jd).
    Parity is intentionally not included.
    """
    Z_parent: int
    A_parent: int
    Z_daughter: int
    A_daughter: int
    Q_alpha_MeV: float
    j_parent: float
    j_daughter: float


# -----------------------------
# Half-life formulae (log10 seconds)
# -----------------------------

def log10T_song(x: AlphaDecayInput, l: Optional[int] = None) -> float:
    
    if x.Q_alpha_MeV <= 0:
        raise ValueError("Q_alpha must be > 0 MeV.")
    if l is None:
        l = lmin_without_parity(x.j_parent, x.j_daughter)

    Z_parent = x.Z_parent
    N_parent = x.A_parent - x.Z_parent

    a, b, heo, hoo = (SONG_COEF[k] for k in ["a", "b", "heo", "hoo"])
    h=0
    r=even_odd_class(Z_parent,N_parent)
    if r in ['eo','oe']:
        h=heo
    elif r=='oo':
        h=hoo

    term_poly = (a+Z_parent+l)*x.Q_alpha_MeV**(-1/2)+b+h

    return term_poly 


# -----------------------------
# Convert to rates
# -----------------------------
def half_life_seconds_from_log10(log10T: float) -> float:
    return 10.0 ** log10T


def decay_constant_from_half_life(T12_s: float) -> float:
    if T12_s <= 0:
        raise ValueError("Half-life must be > 0.")
    return LN2 / T12_s


def alpha_decay_rate(
    x: AlphaDecayInput,
    l: Optional[int] = None,
) -> dict:
    """
    Returns:
      - l_used
      - log10T_sec
      - T12_sec
      - lambda_1_per_s
    """
    log10T = log10T_song(x, l=l)
    l_used = l if l is not None else lmin_without_parity(x.j_parent, x.j_daughter)
    if log10T!='ERROR':
        T12 = half_life_seconds_from_log10(log10T)
        lam = decay_constant_from_half_life(T12)
        return {"l_used": l_used, "log10T_sec": log10T, "T12_sec": T12, "lambda_1_per_s": lam}
    else:
        return {'lambda_1_per_s':'ERROR'}
def format_r1_block(parent,daughter,q_mev,jp,jd,formula) -> str:
    """
    Writes a 3-line REACLIB-R1-like block:
      line1: parent daughter (padded) + source + Q
      line2: a0..a3
      line3: a4..a6
    For constant rate: rate(T9)=exp(a0), set a0=ln(lam), others 0.
    """
  
    decay=AlphaDecayInput(Z_parent=parent.Z,
                          A_parent=parent.A,
                          Z_daughter=daughter.Z,
                          A_daughter=daughter.A,
                          Q_alpha_MeV=q_mev,
                          j_parent=jp,
                          j_daughter=jd)


    # Protect: lambda must be >0
    lam=alpha_decay_rate(decay,formula)['lambda_1_per_s']
    if lam != 'ERROR' and lam>0:
        a0 = math.log(lam)
        # line1: keep it whitespace-separated (parsers using split() will work)
        line1 = f"     {parent.short_spec_name:>5}  he4{daughter.short_spec_name:>5}"+" "*23+"wc12w    "+f"{q_mev: .5e}"+" "*10
        line2 = f"{a0: .6e}"+" 0.000000e+00 0.000000e+00 0.000000e+00                      "
        line3 = " 0.000000e+00 0.000000e+00 0.000000e+00                                   "

        return line1 + "\n" + line2 + "\n" + line3 + "\n"
    else:
        return 'ERROR'


   

def write_alpha_decay_file(
    out_path: str | Path,
    *,
    z_min: int = 50,
    z_max:int=118,
    formula:str
) -> None:
    """
    Create a file containing beta- decay rates computed with Zhou (2017) for all nuclides in WinVN.

    Inputs:
      - winvn_path: Winvn_v2.0-like file (must include Z, N, name, mass excess; stable flag helps δN)
      - out_path: output text file of 3-line blocks

    Filters (recommended):
      - Z >= 20
      - δN > 5 (i.e., N - Nstable(Z) >= 6)
      - Qβ > 0
      - daughter exists in network
    """
    nucs=pd.read_csv(r'winvne_v2.0.dat',skiprows=lambda x: not (x>7854 and (x-7855)%4==0),delim_whitespace=True,names=['name','A','Z','N','spin','Mass excess (Mev)','source'])
   
    

    out_path = Path(out_path)
    n_written = 0
    n_skipped = 0
    by_ZN={}
    for i in range(len(nucs)):
        by_ZN[(nucs['Z'][i], nucs['N'][i])]=[nucs['name'][i],nucs['Mass excess (Mev)'][i],nucs['spin'][i]]

    with out_path.open("w", encoding="utf-8") as f:
        f.write("2"+" "*73+"\n")
        f.write(" "*74+"\n")
        f.write(" "*74+"\n")

        for i in range(len(nucs)):
            p=pyna.nucdata.nucleus.Nucleus(nucs['name'][i])
            if nucs['Z'][i] < z_min or nucs['Z'][i]>z_max:
                continue
            if p.tau=='stable':
                continue
            

            # Daughter: (Z+1, N-1)
            dkey = (nucs['Z'][i] -2, nucs['N'][i] -2)
            d = by_ZN.get(dkey)
            if d is None:
                n_skipped += 1
                continue

            # Qalpha from nuclear mass excess differences
            Qa = nucs["Mass excess (Mev)"][i]-d[1]-2.425
            if Qa <= 0.0:
                n_skipped += 1
                continue

            block = format_r1_block(p,pyna.nucdata.nucleus.Nucleus(d[0]),Qa,nucs['spin'][i],d[2],formula)
            if block!='ERROR':
                f.write(block)
                n_written += 1
            else:
                n_skipped += 1

    print(f"Output: {out_path}")
    print(f"Written alpha- rates: {n_written}")
    print(f"Skipped (invalid log arg etc.): {n_skipped}")

write_alpha_decay_file('SONG_alpha_R1',formula='SONG')


C:\Users\Diego Hernandez\AppData\Local\Temp\ipykernel_4048\3823141811.py:266: RuntimeWarning: overflow encountered in scalar power
  return 10.0 ** log10T


Output: NMSF2021_alpha_decay_R1
Written alpha- rates: 2940
Skipped (invalid log arg etc.): 2554
Output: NMMF2021_alpha_decay_R1
Written alpha- rates: 2109
Skipped (invalid log arg etc.): 3385


C:\Users\Diego Hernandez\AppData\Local\Temp\ipykernel_4048\3823141811.py:266: RuntimeWarning: overflow encountered in scalar power
  return 10.0 ** log10T


Output: SONG_alpha_decay_R1
Written alpha- rates: 2925
Skipped (invalid log arg etc.): 2569


C:\Users\Diego Hernandez\AppData\Local\Temp\ipykernel_4048\3823141811.py:266: RuntimeWarning: overflow encountered in scalar power
  return 10.0 ** log10T


Output: ROYER_alpha_decay_R1
Written alpha- rates: 2922
Skipped (invalid log arg etc.): 2572
